# Actividad 1 · Del relato a los datos: presentación del NSFG

**Recorrido elegido:** SQL + Python  
**Tiempo estimado:** 1 pomodoro (20 minutos)

> Realizá solamente uno de los dos recorridos equivalentes disponibles: **SQL + Python** o **SQL + R**.

En esta actividad vas a distinguir una afirmación anecdótica de una pregunta que puede investigarse con datos, reconocer el contexto del dataset NSFG e identificar su unidad de análisis.


## ¿Los primeros bebés tienden a llegar tarde?

Es posible que hayas escuchado afirmaciones como “el primer hijo siempre tarda más”. Estas expresiones se apoyan en **evidencia anecdótica**: experiencias personales que pueden ser valiosas, pero que no alcanzan para sostener una conclusión general.

Algunas limitaciones habituales son:

- **Pocas observaciones:** una experiencia individual no representa necesariamente a una población.
- **Sesgo de selección:** suelen compartirse más los casos llamativos.
- **Sesgo de confirmación:** recordamos mejor los casos que coinciden con lo que ya creemos.
- **Imprecisión:** los recuerdos pueden redondearse o modificarse con el tiempo.

Para avanzar necesitamos una pregunta clara y un conjunto de datos adecuado.


## La Encuesta Nacional de Crecimiento Familiar (NSFG)

La *National Survey of Family Growth* recopila información sobre vida familiar, embarazos y salud. En este recorrido utilizaremos datos del Ciclo 6, relevado en Estados Unidos entre 2002 y 2003.

| Concepto | En este caso |
|---|---|
| **Población** | Personas comprendidas por el diseño de la encuesta |
| **Muestra** | Personas efectivamente seleccionadas y encuestadas |
| **Unidad de análisis del archivo** | Cada embarazo informado por una encuestada |
| **Registro** | Una fila de la tabla `embarazos` |

La unidad de análisis es especialmente importante: varias filas pueden corresponder a embarazos diferentes de una misma persona.


## Preparación automática

Ejecutá la celda siguiente sin modificarla. Su única función es descargar y preparar los datos y dejar disponible la tabla `embarazos` en DuckDB. Este procedimiento técnico no forma parte de la evaluación.


In [ ]:
#@title Preparación automática del entorno (ejecutar sin modificar)
import importlib.util
import subprocess
import sys
from pathlib import Path
from urllib.request import urlretrieve

if importlib.util.find_spec("duckdb") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "duckdb"])

import duckdb
import numpy as np
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/AllenDowney/ThinkStats/v3/data/2002FemPreg.dat.gz"
DATA_PATH = Path("2002FemPreg.dat.gz")
if not DATA_PATH.exists():
    urlretrieve(DATA_URL, DATA_PATH)

colspecs = [
    (0, 12), (12, 14), (56, 58), (58, 60), (274, 276),
    (276, 277), (277, 279), (283, 287), (422, 440),
]
names = [
    "caseid", "pregordr", "birthwgt_lb", "birthwgt_oz", "prglngth",
    "outcome", "birthord", "agepreg", "finalwgt",
]

embarazos = pd.read_fwf(
    DATA_PATH, names=names, colspecs=colspecs, compression="gzip"
)
embarazos["birthwgt_lb"] = embarazos["birthwgt_lb"].replace([51, 97, 98, 99], np.nan)
embarazos["birthwgt_oz"] = embarazos["birthwgt_oz"].replace([97, 98, 99], np.nan)
embarazos["totalwgt_lb"] = embarazos["birthwgt_lb"] + embarazos["birthwgt_oz"] / 16
embarazos["totalwgt_kg"] = (embarazos["totalwgt_lb"] * 0.45359237).round(3)
embarazos["agepreg"] = embarazos["agepreg"] / 100
embarazos = embarazos[[
    "caseid", "pregordr", "prglngth", "outcome", "birthord",
    "birthwgt_lb", "birthwgt_oz", "totalwgt_lb", "totalwgt_kg",
    "agepreg", "finalwgt",
]]

con = duckdb.connect(":memory:")
con.register("embarazos_df", embarazos)
con.execute("CREATE OR REPLACE TABLE embarazos AS SELECT * FROM embarazos_df")
print(f"Entorno preparado: {len(embarazos):,} registros disponibles en la tabla 'embarazos'.")


## Primer vistazo con DuckDB y SQL

**DuckDB** es un motor de base de datos que permite consultar tablas mediante SQL sin instalar un servidor. En ambos recorridos, la pregunta dirigida a los datos se expresa con la misma consulta SQL; Python se utiliza para preparar el entorno y mostrar los resultados.


In [ ]:
con.sql("""
SELECT * FROM embarazos LIMIT 5
""")


## Variables principales

| Variable | Descripción |
|---|---|
| `caseid` | Identificador de la encuestada |
| `pregordr` | Orden del embarazo informado |
| `prglngth` | Duración del embarazo en semanas |
| `outcome` | Resultado del embarazo |
| `birthord` | Orden de nacimiento entre los nacidos vivos |
| `totalwgt_kg` | Peso total al nacer, expresado en kilogramos |
| `agepreg` | Edad de la madre al finalizar el embarazo |
| `finalwgt` | Peso estadístico de la observación |


Contemos los registros y las personas distintas. La diferencia entre ambas cantidades permite comprobar que la tabla registra **embarazos**, no personas.


In [ ]:
con.sql("""
SELECT
    COUNT(*) AS total_embarazos,
    COUNT(DISTINCT caseid) AS total_encuestadas
FROM embarazos
""")


## Cierre

El dataset contiene **13.593 embarazos**. Antes de analizar distribuciones o comparar grupos, es necesario comprender qué representa cada fila, de dónde provienen los datos y qué significan sus variables.

**Bibliografía:** Downey, A. B. (2025). *Think Stats* (3.ª ed.), capítulo 1.


In [ ]:
con.close()
